In [ ]:
# Import modules

# Sci computing
import numpy as np
import scipy as sp
import seawater as sw
import scipy.sparse.linalg as sla
from contourpy import contour_generator
import pyfftw 

# Importing bathymetry (tiff)
import rasterio as geo

# Parallel comupting
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar

# For Data
import netCDF4 as nc
import xarray as xr

# Plotting stuff
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as grdspc
import cmocean as cm

# Gen stuff
from datetime import date
today = date.today()

In [ ]:
# For plotting; have the bathymetry of the gulf

ds_gulf = geo.open('/home/justin_cooke_uri_edu/DeepCyclones/gulf.tiff')
#ds_gulf = geo.open('../gulf.tiff')
img = ds_gulf.read(1)

(gm,gn) = np.shape(img)
gulf_lon = np.linspace(-90,-82,gn)
gulf_lat = np.linspace(22,28,gm)

# To get the aspect ratio right
ymid = np.mean(gulf_lat)
ymid_rad = ymid*np.pi/180

In [ ]:
ds_eref = xr.open_mfdataset('./hycom_data/hycom_etaref_*.nc',combine='nested',concat_dim='MT')
ds_ssh = xr.load_dataset('./hycom_data/hycom_ssh_filtered.nc')

In [ ]:
# Load the variables we care about
eref = ds_eref['eta_ref']
ssh = ds_ssh['ssh']
ssh_demean = ssh - xr.DataArray.mean(ssh,dim=["Latitude","Longitude"],skipna=True)

lon = ds_eref['Longitude']
lat = ds_eref['Latitude']

Nt,Nlat,Nlon = eref.shape

In [ ]:
RadEarth = 6371  # [km]

LC = {
    'length': np.zeros(Nt),
    'west': np.zeros(Nt),
    'north': np.zeros(Nt)
}

for j in range(Nt):

    z_field = ssh_demean.isel(MT=j)

    # Create a ContourGenerator instance once (x, y are 1D coordinate arrays)
    contour_gen = contour_generator(
        x=lon, y=lat, z=z_field,  # z will be passed per iteration
        line_type="Separate"          # ensures each contour is a separate array
    )


    # Generate contours for the j-th SSH slice at level 0.17
    contour_segments = contour_gen.lines(0.17)  # list of Nx2 arrays (x, y)

    curveLength = np.zeros(len(contour_segments))
    curveWest = np.zeros(len(contour_segments))
    curveNorth = np.zeros(len(contour_segments))

    for k, seg in enumerate(contour_segments):
        myX = seg[:,0]
        myY = seg[:,1]

        YCheck = myY <= 22.1
        XCheck = myX >= -86.5

        if (-82 in myX) or (np.any(YCheck) and np.any(XCheck)):
            idx_candidates = np.where(myY >= 23)[0]
            idx_w = idx_candidates[0] if len(idx_candidates) > 0 else None

            curveNorth[k] = np.max(myY)

            if idx_w is None or myY[idx_w] == 90:
                curveWest[k] = 0
            else:
                curveWest[k] = myX[idx_w]

            # Compute great-circle distance along the contour
            this_curve = 0.0
            for i in range(len(myX) - 1):
                phi_1 = np.deg2rad(myX[i])
                phi_2 = np.deg2rad(myX[i + 1])
                lam_1 = np.deg2rad(myY[i])
                lam_2 = np.deg2rad(myY[i + 1])

                delphi = phi_2 - phi_1
                dellam = lam_2 - lam_1

                thisDist = 2 * RadEarth * np.asin( np.sqrt(( 1 - np.cos((delphi)) +
                                                  ((np.cos(phi_1)) * np.cos(phi_2) * 
                                                   (1 - np.cos(dellam) ) )) / 2)) 

                this_curve += thisDist

            curveLength[k] = this_curve

    LC['west'][j] = np.max(np.abs(curveWest)) if len(curveWest) > 0 else 0
    LC['north'][j] = np.max(np.abs(curveNorth)) if len(curveNorth) > 0 else 0
    LC['length'][j] = np.sum(curveLength) if len(curveLength) > 0 else 0


In [ ]:
# CEOF Function

def c_eof(D,NOE=10):
    # This function determines the complex empirical orthogonal functions of a data set contained in matrix D

    # INPUTS:
    # D: Each row is assumed to be a sample; each column a variable. Thus, a column represents a time-series of one variable (or at one point)
    # NOE: Number of Eigenvalues (optional, if non-given then NOE=10)

    # OUTPUTS:
    # V: Vector of real eigenvalues 
    # EOFs: Matrix with complex values, each column represents an EOF
    # EC: EOF coefficients, also called principal component coefficients, i.e., the original time-series transformed to EOF space
    # error: The L2-norm of the reconstruction error of each point

    # Written by Justin Cooke, 2025, based on the MATLAB code written by Martijn Hooimeijer, 1999.

    # n = number of spatial points, m = number of time steps
    (n,m) = np.shape(D)
    q = np.min((n,m))

    # Hilbert transform the D matrix 
    def hilbtrans(X):

        # Now we will enter the frequency domain
        # Conduct the fft, using pyfftw so that we use an fftw library wrapper
        Y = pyfftw.interfaces.scipy_fft.fft(X,axis=0) # Need to set the axis to be zero to go along the first axis

        # Get the shape 
        (p,q) = np.shape(Y)
        N = p 
        # print('N = ',N)
        N2 = np.floor(N/2) - 1 # effect of odd and even # of elements
        N2 = np.array(np.floor(N/2),dtype=np.int16)
        # N2 = int(N2)
        # print('N2 = ',N2)

        # Now, rotate the first half of the matrix
        #print('Rotating the first half of the matrix CCW 90 degrees ...')
        P = np.zeros((p,q),dtype=complex) # Allocate a complex matrix, P, to be filled
        P[0:(N2),:] = Y[0:(N2),:]*1j
        #print('Here is the first half of P: ','\n',P[0:(N2),:])

        N2 = N - N2
        #print('New N2 = ',N2)

        P[N2:N,:] = Y[N2:N,:]/1j
        #print('Second half of P = ','\n',P[N2:N])

        # Now we need to go back to the time domain using ifft
        XH = pyfftw.interfaces.scipy_fft.ifft(P,axis=0,norm='backward')

        # XH will most likely return something that has the same real part as MATLAB, but small differences in the complex part
        
        return XH
    
    # Hilbert transform of the data matrix, D
    DH = hilbtrans(D)

    # Allocate memory for a complex matrix, DC, which is the original matrix, D, plus the hilbert transform of that matrix
    DC = np.zeros((n,m),dtype='complex')
    DC = D + DH*1j

    def EOF2(D,p): # Computes the eigenvalues, EOFs, and EOF Coefficients, input is D and the number of eigenvalues (NOE)
        
        (m,n) = np.shape(D) 
        Ma = np.sum(DC,axis=0)/m # Find the average of each time-series (column)
        DS = DC - np.tile(Ma,(m,1)) # Remove this average to make each time-series have zero-mean
        q = np.min((m,n)) # What is smaller, the time series of the number of data points?

        NOE = min(q,p) # NOE will either be the smallest number b/t the size and the inputted NOE

        if m >= n:
            CE = (DS.conj().T @ DS) / (m-1)
            if np.isscalar(CE):
                (S,EOFs) = sla.eigs(CE,NOE)
            else:
                (S,EOFs) = sla.eigs(A=CE,k=NOE,M=np.eye(n,n),return_eigenvectors=True)

        if m < n:
            CE = (DS @ DS.conj().T) / (m-1)
            (S,E) = sla.eigs(A=CE,k=NOE,M=np.eye(m,m))
            EOFs = DC.conj().T @ E
            for i in range(NOE-1):
                EOFs[:,i]  = EOFs[:,i] / np.linalg.norm(EOFs[:,i])

        V = (np.real(S))

        EC = DS @ EOFs

        diff = (DS - (EC @ EOFs.conj().T))

        error = np.sqrt(np.sum(np.abs(np.pow(diff,2)),axis=0))
        # print(error)
            
        return(V,EOFs,EC,error)

    (V,EOFs,EC,error) = EOF2(DC,NOE)

    return(V,EOFs,EC,error)

In [ ]:
# Low Pass Filter

from scipy.signal import butter, filtfilt

def lowpassfilt(data, filtT, sampT):
    """
    Low-pass filter a 1D data vector using a second-order Butterworth filter.
    Filtering is applied forward and backward to eliminate phase shift.

    Parameters
    ----------
    data : array_like
        Input data vector (1D array). Can be row or column oriented.
    filtT : float
        Filter period (same units as sampT).
    sampT : float
        Sampling period (same units as filtT).

    Returns
    -------
    out : ndarray
        Filtered data vector, same shape as input.
    """

    data = np.asarray(data)
    if data.ndim != 1:
        raise ValueError("Input must be a 1D vector, not an array.")

    # Check if input was a row vector (for shape preservation)
    was_row = data.ndim == 1 and data.shape[0] < data.shape[-1]

    # Remove linear trend between endpoints
    n = len(data)
    x = np.arange(1, n + 1)
    p = np.polyfit([1, n], [data[0], data[-1]], 1)
    lineartrend = np.polyval(p, x)
    detrended = data - lineartrend

    # Check for NaNs
    if np.isnan(detrended).any():
        raise ValueError("Data contains NaNs; remove them before filtering.")

    # Butterworth filter setup
    # Wn is the normalized cutoff frequency (Nyquist = 0.5 / sampT)
    Wn = (2.0 / filtT) * sampT
    if Wn >= 1:
        raise ValueError("Cutoff frequency too high; ensure filtT > 2 * sampT.")
    b, a = butter(2, Wn)

    # Zero-phase filtering
    filtered = filtfilt(b, a, detrended)

    # Add linear trend back
    out = filtered + lineartrend

    # Preserve input orientation
    if was_row:
        out = out.reshape(1, -1)
    else:
        out = out.reshape(-1)

    return out


In [ ]:
# Create the D Matrix

# Get eta ref values
eref_vals = eref.values

# Create a mask containing all the points that are not NaNs
mask = ~np.isnan(eref_vals[0,:,:])
idx = np.where(mask)

# Create D
D = eref_vals[:,idx[0],idx[1]]
D = np.moveaxis(D,0,1)

lp_D = np.empty_like(D)

fc = 10.0
sc = 1.0

for i in range(D.shape[0]):
    lp_D[i,:] = lowpassfilt(D[i,:],fc,sc)

In [ ]:
# Do the CEOF

NOE=10

[V,EOFs,EC,_] = c_eof(np.transpose(lp_D),NOE)

In [ ]:
max_mode = 4

mode_key = ['M1','M2','M3','M4']

z1 = {}
zt = {}
norm_amp = {}
phase = {}

for n,mode in enumerate(mode_key):
    factor = 1 / np.max(np.abs(EOFs[:,n]))
    z1[mode] = np.abs(EOFs[:,n]) * factor
    zt[mode] = np.abs(EC[:,n]) / factor

# Reconstruct maps in lat lon space

for n,mode in enumerate(mode_key):
    shell_amp = np.full((Nlat,Nlon),np.nan)
    shell_phase = np.full((Nlat,Nlon),np.nan)

    tempvar = EOFs[:,n]
    tempcoeff = EC[:,n]
    tempvar = 180 - np.angle(tempvar) * 180 / np.pi

    shell_amp[idx[0],idx[1]] = z1[mode]
    norm_amp[mode] = shell_amp

    shell_phase[idx[0],idx[1]] = tempvar
    phase[mode] = shell_phase

phase_masked = {}

for n,mode in enumerate(mode_key):
    
    phase_masked[mode] = np.ma.masked_where(norm_amp[mode] <= 0.2,phase[mode])


In [ ]:
# Percent for each mode
percentvar = np.empty(len(mode_key))

for n,_ in enumerate(mode_key):
    percentvar[n] = V[n] / np.sum(V)
    percentvar[n] = np.round(percentvar[n],3) * 100

In [ ]:
# Smooth coefficients

def movmean(data,window_size):

    smoothed_data = np.cumulative_sum(data,dtype=float)
    smoothed_data[window_size:] = smoothed_data[window_size:] - smoothed_data[:-window_size]
    return smoothed_data[window_size - 1:] / window_size

smooth_coeffs = {}
for n,mode in enumerate(mode_key):
    smooth_coeffs[mode] = movmean(zt[mode],60)

In [ ]:
#-------------------------------------------------- PLOT AMP + PHASE + COEFFS

fig = plt.figure(figsize=(20,10))
gs0 = fig.add_gridspec(3,1)
gs1 = grdspc.GridSpecFromSubplotSpec(1,4,subplot_spec=gs0[0])
gs2 = grdspc.GridSpecFromSubplotSpec(1,4,subplot_spec=gs0[1])
gs3 = grdspc.GridSpecFromSubplotSpec(1,1,subplot_spec=gs0[2])


for row in range(1):
    for col in range(4):
        ax = fig.add_subplot(gs1[row,col])
    
        cb1 = ax.contourf(lon,lat,norm_amp[mode_key[col]],cmap=cm.cm.ice_r,levels=np.linspace(0,1.0,11),extend='both')
        ax.contour(gulf_lon,gulf_lat,np.flipud(img),colors='black',linestyles='-',levels=[-3500,-3000,-2500,-2000])
        ax.set_aspect(np.cos(ymid))
        ax.set_xlabel('Longitude [$^\circ$W]',fontsize=14)
        ax.set_ylabel('Latitude [$^\circ$N]',fontsize=14)
        fig.colorbar(cb1,shrink=0.75)
        ax.set_title(f'{mode_key[col]}: {percentvar[col]}$\%$',fontsize=18)


       
for row in range(1):
    for col in range(4):
        ax = fig.add_subplot(gs2[row,col])
        cb2 = ax.contourf(lon,lat,phase_masked[mode_key[col]],cmap=cm.cm.phase,levels=np.linspace(0,360,9),extend='both')
        ax.contour(gulf_lon,gulf_lat,np.flipud(img),colors='black',linestyles='-',levels=[-3500,-3000,-2500,-2000])
        ax.set_aspect(np.cos(ymid))
        ax.set_xlabel('Longitude [$^\circ$W]',fontsize=14)
        ax.set_ylabel('Latitude [$^\circ$N]',fontsize=14)
        fig.colorbar(cb2,shrink=0.75)

ax = fig.add_subplot(gs3[0])
for n,mode in enumerate(mode_key):
    ax.plot(zt[mode],label=f'{mode}: {percentvar[n]}$\%$')
ax.set_xlim(0,Nt)
ax.hlines(np.mean(zt['M1'])+np.std(zt['M1']),0,Nt)

yr_ticks = [i for i in range(0,Nt,365)]
yr_labels = [i for i in range(0,19,1)]

ax.set_xticks(yr_ticks)
ax.set_xticklabels(yr_labels)

ax.legend(ncols=4,loc='upper center')


In [ ]:
# Conditionally average the modes 
# Find the time periods where each Mode peaks 

days = np.arange(Nt)

cond_avg = {}

for n,mode in enumerate(mode_key):
    cond_avg[mode] = {}
    cond_avg[mode]['ts'] = smooth_coeffs[mode] # coefficient time-series
    cond_avg[mode]['mean'] = np.mean(zt[mode]) # mean of the coefficient time-series
    cond_avg[mode]['std'] = np.std(zt[mode]) # standard dev. of the coefficient time-series
    th_mask = (zt[mode] >= (np.mean(zt[mode]) + np.std(zt[mode]))) # make a mask where the time-series is greater than the mean plus one standard deviation
    idx_th = np.where(th_mask) # indices of the mask
    cond_avg[mode]['th'] = days[idx_th] # recorded as the threshold days
    cond_avg[mode]['length'] = LC['length'][idx_th] # LC length only using the days that meet the threshold
    cond_avg[mode]['north'] = LC['north'][idx_th] # LC northern extension only using the days that meet the threshold
    cond_avg[mode]['eref'] = eref.values[idx_th,:,:] # Eta Ref values only using the days that meet the threshold



In [ ]:
# now we need the indices for the time-series chunks
# initialize
for mode in mode_key:
    ct = 0
    keep_me = []

    for n in range(len(cond_avg[mode]['th'])-1):
        pt_a = cond_avg[mode]['th'][n+1]
        pt_b = cond_avg[mode]['th'][n]

        diff_ab = pt_a - pt_b

        if diff_ab != 1:
            keep_me.append(n + 1)
            
            ct += 1
    
    
    cond_avg[mode]['idx']  = np.array(keep_me)

In [ ]:
# Find LC separation times not counting detachments

count_me = 0
sep_pt_400 = []

for n in range(Nt-1):
    pt1 = LC['length'][n]
    pt2 = LC['length'][n+1]

    pt_diff = pt1 - pt2

    if pt_diff >= 200:
        j = 1
        while j <= 40:
            pt_diff_test = pt1 - LC['length'][n+j+1]
            if pt_diff_test <= 400:
                break
            else:
                j += 1
                continue

        if j > 40:
            sep_pt_400.append(n)
            count_me += 1

when_separation = np.array(sep_pt_400)

In [ ]:
# Plot the LC Length with bars for the peak times of each coefficient

from matplotlib.patches import Rectangle

fig,ax = plt.subplots(figsize=(30,5))
ax.plot(days,LC['length'],color='k',linewidth=2)
ax.set_ylim(250, 2000)

ax.set_xticks(yr_ticks)
ax.set_xticklabels(yr_labels)
ax.set_xlabel('Years',fontsize=16)
ax.set_ylabel('LC Length [km]',fontsize=16)
ax.tick_params(axis='both', which='major', labelsize=14)
# Mode 1
for i in range(len(cond_avg['M1']['idx'])-1):
    if i == 0:
        pt1 = 0
        pt2 = cond_avg['M1']['th'][cond_avg['M1']['idx'][i]-1]
        rec_width = pt2-pt1
    else:
        pt1 = cond_avg['M1']['th'][cond_avg['M1']['idx'][i]]
        pt2 = cond_avg['M1']['th'][cond_avg['M1']['idx'][i+1]-1]
        rec_width = pt2-pt1

    ax.add_patch(Rectangle((pt1,1900),width=rec_width,height=100,color='xkcd:muted blue'))

# Mode 2
for i in range(0,len(cond_avg['M2']['idx'])-1,2):
    if i == 0:
        pt1 = 0
        pt2 = cond_avg['M2']['th'][cond_avg['M2']['idx'][i]-1]
        rec_width = pt2-pt1
    else:
        pt1 = cond_avg['M2']['th'][cond_avg['M2']['idx'][i]]
        pt2 = cond_avg['M2']['th'][cond_avg['M2']['idx'][i+1]-1]
        rec_width = pt2-pt1

    ax.add_patch(Rectangle((pt1,1800),width=rec_width,height=100,color='xkcd:rusty orange'))

# Mode 3
for i in range(0,len(cond_avg['M3']['idx'])-1,2):
    if i == 0:
        pt1 = 0
        pt2 = cond_avg['M3']['th'][cond_avg['M3']['idx'][i]-1]
        rec_width = pt2-pt1
    else:
        pt1 = cond_avg['M3']['th'][cond_avg['M3']['idx'][i]]
        pt2 = cond_avg['M3']['th'][cond_avg['M3']['idx'][i+1]-1]
        rec_width = pt2-pt1
   
    ax.add_patch(Rectangle((pt1,1700),width=rec_width,height=100,color='xkcd:marigold'))

for n in range(len(when_separation)):
    ax.vlines(when_separation[n],250,2000,color='xkcd:electric purple')
ax.set_xlim(0,Nt)

ax.text(Nt-250,1920,'Mode 1',fontsize=16)
ax.text(Nt-250,1820,'Mode 2',fontsize=16)
ax.text(Nt-250,1720,'Mode 3',fontsize=16)

fig.savefig(f'./Figures/LC_Length_ColorCoded_{today}.svg',format='svg')